# Train YOLOv5 human detector on Google Colab

Runtime > Change runtime type > **GPU (T4)** before running.

Steps: data -> split train/val/test -> train -> validate -> test -> download `best.pt`.

## 1. Setup

In [ ]:
!git clone https://github.com/chemsoubensalah/real_time-_object_detection.git project
!git clone https://github.com/ultralytics/yolov5
%pip install -q -r yolov5/requirements.txt
import torch; print('GPU:', torch.cuda.is_available())

## 2. Dataset

**Option A** - automatic, COCO person images (no manual labelling).

**Option B** - your own phone images labelled with Roboflow/LabelImg: upload a zip containing `images/` and `labels/` and unzip it to `data/raw`.

You can do both, the images end up in the same folder.

In [ ]:
# Option A
!python project/scripts/download_coco_person.py --num 1500 --out data/raw

In [ ]:
# Option B (uncomment)
# from google.colab import files
# up = files.upload()            # my_images.zip with images/ and labels/
# !unzip -q -o my_images.zip -d data/raw

## 3. Split into train / val / test (80/10/10)

In [ ]:
!python project/scripts/split_dataset.py --src data/raw --dst data/dataset --names person
!cat data/dataset/data.yaml

## 4. Training

Starts from pretrained `yolov5s.pt` (transfer learning). Try `yolov5n.pt` for more FPS or `yolov5m.pt` for more accuracy.

In [ ]:
!cd yolov5 && python train.py --img 640 --batch 16 --epochs 100 --data ../data/dataset/data.yaml --weights yolov5s.pt --name person --exist-ok

## 5. Validation curves (loss, precision, recall, mAP)

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    display(Image(f'yolov5/runs/train/person/{f}', width=900))

## 6. Test set evaluation (images never seen during training)

In [ ]:
!cd yolov5 && python val.py --weights runs/train/person/weights/best.pt --data ../data/dataset/data.yaml --task test --name person_test --exist-ok

## 7. Try on test images / a video

In [ ]:
!cd yolov5 && python detect.py --weights runs/train/person/weights/best.pt --source ../data/dataset/images/test --name person_detect --exist-ok
import glob
for p in sorted(glob.glob('yolov5/runs/detect/person_detect/*.jpg'))[:6]:
    display(Image(p, width=500))

## 8. Download the trained model

Then run on your computer with the phone camera:
```
python scripts/detect_realtime.py --weights best.pt --source http://PHONE_IP:8080/video
```

In [ ]:
from google.colab import files
files.download('yolov5/runs/train/person/weights/best.pt')